# MASLD-LLM · 03 — Fine-tuning on Colab (Drive-synced edition)



Trains the model ladder from your Drive-synced folders. Sync mounts at

`/content/drive/Othercomputers/My Mac/` (read-only); checkpoints go to the

writable fallback if needed.



## Session 5 (smoke test): runtime = **L4**, model = **Qwen3-0.6B**



Paper recipe: LoRA r32 α32, ChatML, 3 epochs, assistant-only loss, neat packing.


## 0 · Mount + auto-discover


In [16]:
from google.colab import drive
drive.mount('/content/drive')
import os

SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']

def find_dir(target_name, search_roots=None, depth=6):
    if search_roots is None:
        search_roots = SEARCH_ROOTS
    matches = []
    for root in search_roots:
        direct = os.path.join(root, target_name)
        if os.path.isdir(direct):
            matches.append(direct)
    if matches:
        return matches
    def walk(path, d):
        if d > depth:
            return
        try:
            for entry in os.scandir(path):
                if entry.is_dir(follow_symlinks=False):
                    if entry.name == target_name:
                        matches.append(entry.path)
                    walk(entry.path, d + 1)
        except (PermissionError, OSError):
            pass
    for root in search_roots:
        if os.path.exists(root):
            walk(root, 0)
    return matches

cands = find_dir('masld-llm')
MASLD_LLM = min(cands, key=len) if cands else None
assert MASLD_LLM, 'masld-llm not found under Othercomputers/My Mac or MyDrive'
PROMPTS = os.path.join(MASLD_LLM, 'prompts')
print('PROMPTS =', PROMPTS)
print('corpora:', sorted(f for f in os.listdir(PROMPTS) if f.endswith('.jsonl.gz')))

def is_writable(path):
    try:
        probe = os.path.join(path, '.write_test')
        open(probe, 'w').close(); os.remove(probe)
        return True
    except Exception:
        return False

CKPT = (os.path.join(MASLD_LLM, 'checkpoints') if is_writable(MASLD_LLM)
        else '/content/drive/MyDrive/masld-llm-outputs/checkpoints')
os.makedirs(CKPT, exist_ok=True)
print('CKPT (writable) =', CKPT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROMPTS = /content/drive/Othercomputers/My Mac/masld-llm/prompts
corpora: ['MB-10xx_liver_train.jsonl.gz', 'MB-11xx_external_train.jsonl.gz', 'MB-20xx_liver_test.jsonl.gz', 'MB-30xx_dnam_test.jsonl.gz', 'MB-30xx_dnam_train.jsonl.gz', 'MB-61xx_msigdb_train.jsonl.gz', 'MB-62xx_drugs_train.jsonl.gz', 'MB-63xx_drugcentral_affinity_train.jsonl.gz', 'MB-70xx_dge_direction_train.jsonl.gz', 'MB-8xxx_fujiwara_train.jsonl.gz', 'MB-9xxx_fujiwara_KU_test.jsonl.gz']
CKPT (writable) = /content/drive/Othercomputers/My Mac/masld-llm/checkpoints


## 1 · Install + convert all seven train corpora


In [17]:
%cd /content
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip -q install -e ".[torch,metrics]" bitsandbytes peft transformers accelerate

import gzip, json, os

def to_alpaca(in_path, out_path):
    rows = []
    with gzip.open(in_path, 'rt') as f:
        first = json.loads(f.readline())
        if '__meta__' not in first:
            rows.append(first)
        for l in f:
            rows.append(json.loads(l))
    out = [dict(instruction=r['prompt'], output=str(r['answer'])) for r in rows]
    json.dump(out, open(out_path, 'w'), indent=0)
    print(out_path, len(out))

to_alpaca(f'{PROMPTS}/MB-10xx_liver_train.jsonl.gz',      '/content/sft_liver.json')
to_alpaca(f'{PROMPTS}/MB-11xx_external_train.jsonl.gz',   '/content/sft_external.json')
to_alpaca(f'{PROMPTS}/MB-61xx_msigdb_train.jsonl.gz',     '/content/sft_knowledge.json')
to_alpaca(f'{PROMPTS}/MB-62xx_drugs_train.jsonl.gz',      '/content/sft_drugs.json')
to_alpaca(f'{PROMPTS}/MB-63xx_drugcentral_affinity_train.jsonl.gz', '/content/sft_drugs2.json')
to_alpaca(f'{PROMPTS}/MB-70xx_dge_direction_train.jsonl.gz', '/content/sft_dge.json')
to_alpaca(f'{PROMPTS}/MB-8xxx_fujiwara_train.jsonl.gz',   '/content/sft_fujiwara.json')
to_alpaca(f'{PROMPTS}/MB-30xx_dnam_train.jsonl.gz',       '/content/sft_dnam.json')

/content
fatal: destination path 'LLaMA-Factory' already exists and is not an empty directory.
/content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
/content/sft_liver.json 3928
/content/sft_external.json 5575
/content/sft_knowledge.json 2701
/content/sft_drugs.json 30000
/content/sft_drugs2.json 39632
/content/sft_dge.json 8088
/content/sft_fujiwara.json 204
/content/sft_dnam.json 73


In [18]:
# hierarchical-uniform domain sampling -> ~60k-prompt mixture
import json, os, random
rs = random.Random(20260929)

DOMAIN_SHARE = {
  'liver':    (['/content/sft_liver.json', '/content/sft_external.json', '/content/sft_fujiwara.json'], 0.35),
  'knowledge':(['/content/sft_knowledge.json', '/content/sft_dge.json'], 0.15),
  'drugs':    (['/content/sft_drugs.json', '/content/sft_drugs2.json'], 0.25),
  'dnam':     (['/content/sft_dnam.json'], 0.05),
  'tahoe':    (['/content/sft_tahoe.json'], 0.20),   # absent until notebook 01 builds it
}
TARGET = 60_000

pool = {}
for k, (fps, share) in DOMAIN_SHARE.items():
    rows = []
    for fp in fps:
        if os.path.exists(fp):
            rows += json.load(open(fp))
    pool[k] = rows
    if not rows:
        print(f'  WARNING: domain {k!r} empty (files missing) - share redistributed')
    print(k, len(rows))

out = []
for k, (fps, share) in DOMAIN_SHARE.items():
    rows = pool.get(k) or []
    n = int(TARGET * share)
    if len(rows) > n:
        rows = rs.sample(rows, n)
    out += rows
rs.shuffle(out)
json.dump(out, open('/content/sft_multitask.json', 'w'), indent=0)
print('mixture size:', len(out))

liver 9707
knowledge 10789
drugs 69632
dnam 73
tahoe 0
mixture size: 33780


## 2 · Register dataset + 0.6B config


In [19]:
ds_entry = '''{
  "masld_sft": {
    "file_name": "/content/sft_multitask.json",
    "columns": {"prompt": "instruction", "response": "output"}
  }
}'''
open('data/dataset_info.json', 'w').write(ds_entry)

yaml_06b = '''
model_name_or_path: Qwen/Qwen3-0.6B
trust_remote_code: true
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 32
lora_alpha: 32
lora_dropout: 0.0
dataset: masld_sft
template: qwen
cutoff_len: 4096
packing: true
neat_packing: true
max_samples: 100000
overwrite_cache: true
preprocessing_num_workers: 16
output_dir: ''' + CKPT + '''/qwen3-0.6b-masld-sft
logging_steps: 10
save_steps: 0.25
plot_loss: true
per_device_train_batch_size: 1
gradient_accumulation_steps: 32
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
weight_decay: 0.01
bf16: true
gradient_checkpointing: true
enable_liger_kernel: true
'''
open('/content/qwen06b_sft.yaml', 'w').write(yaml_06b)
print('config written; CKPT =', CKPT)
print('OOM fixes: cutoff_len 8192->4096, batch 2->1, grad_accum 16->32 (same effective 32),')
print('           enable_liger_kernel: true (fused CE - the vocab logits tensor was the 9.3GB OOM)')

config written; CKPT = /content/drive/Othercomputers/My Mac/masld-llm/checkpoints
OOM fixes: cutoff_len 8192->4096, batch 2->1, grad_accum 16->32 (same effective 32),
           enable_liger_kernel: true (fused CE - the vocab logits tensor was the 9.3GB OOM)


In [20]:
import os

# Fix CUDA version mismatch between PyTorch (cu130) and TorchAudio (cu128)
!pip uninstall -y torchaudio
!pip install torchaudio --index-url https://download.pytorch.org/whl/cu130

# Liger kernels (fused cross-entropy) - required to avoid the 9.3GB vocab-logits OOM
!pip -q install liger-kernel

# Session 5 smoke test: watch loss drop 4-5 -> <2 within 50 steps
!DISABLE_VERSION_CHECK=1 python -m llamafactory.cli train /content/qwen06b_sft.yaml

Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Looking in indexes: https://download.pytorch.org/whl/cu130
  Using cached https://download-r2.pytorch.org/whl/cu130/torchaudio-2.11.0%2Bcu130-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached https://download-r2.pytorch.org/whl/cu130/torchaudio-2.11.0%2Bcu130-cp313-cp313-manylinux_2_28_x86_64.whl (1.7 MB)
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/_ops.py", line 1503, in load_library
    ctypes.CDLL(path)
    ~~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/ctypes/__init__.py", line 361, in __init__
    self._handle = self._load_library(name, mode, handle, winmode)
                   ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/ctypes/__init__.py", line 403, in _load_library
    return _dlopen(name, mode)
OSError: libcudart.so.13: cannot open shared objec

## 3 · Evaluate on the frozen test (first C2-contrast row)


In [21]:
# Merge the LoRA adapter into the base model BEFORE installing vLLM.
# Primary: llamafactory export (yaml, output captured). Fallback: direct PeFT
# merge_and_unload() - no llamafactory dependency, works with any checkpoint dir.
import subprocess, sys, os, glob, shutil

CKPT_SFT = f'{CKPT}/qwen3-0.6b-masld-sft'

adapter = None
if os.path.exists(os.path.join(CKPT_SFT, 'adapter_model.safetensors')):
    adapter = CKPT_SFT
else:
    ckpts = sorted(glob.glob(f'{CKPT_SFT}/checkpoint-*'),
                   key=lambda p: int(p.rsplit('-', 1)[-1]))
    if ckpts:
        adapter = ckpts[-1]
assert adapter, f'No adapter found in {CKPT_SFT} - re-run cell 8 (training) first.'
if adapter != CKPT_SFT:
    print('NOTE: root adapter missing => training did not finish.')
    print('      Merging the LATEST checkpoint. Check cell 8 output to see why it stopped;')
    print('      re-run cell 8 for the final adapter if it crashed.')
print('merging adapter:', adapter)

MERGED = '/content/qwen3-0.6b-masld-merged'   # local disk: fast, avoids Drive stalls

# ---------- primary: llamafactory export (errors now visible) ----------
export_yaml = f'''
model_name_or_path: Qwen/Qwen3-0.6B
adapter_name_or_path: {adapter}
template: qwen
finetuning_type: lora
export_dir: {MERGED}
export_size: 2
export_device: cpu
export_legacy_format: false
'''
open('/content/export_06b.yaml', 'w').write(export_yaml)

ok_export = False
try:
    env = dict(os.environ, DISABLE_VERSION_CHECK='1')
    result = subprocess.run(
        [sys.executable, '-m', 'llamafactory.cli', 'export', '/content/export_06b.yaml'],
        env=env, capture_output=True, text=True, timeout=1800)
    if result.stdout:
        print('--- llamafactory export stdout (tail) ---')
        print(result.stdout[-2500:])
    if result.returncode == 0 and os.path.exists(f'{MERGED}/config.json'):
        ok_export = True
    else:
        print('--- llamafactory export FAILED, stderr (tail) ---')
        print(result.stderr[-2500:])
except Exception as e:
    print('llamafactory export raised:', e)

# ---------- fallback: direct PeFT merge (always works) ----------
if not ok_export:
    print('=== falling back to direct PeFT merge_and_unload() ===')
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(
        'Qwen/Qwen3-0.6B', torch_dtype=torch.bfloat16)
    model = PeftModel.from_pretrained(base, adapter)
    model = model.merge_and_unload()
    model.save_pretrained(MERGED, safe_serialization=True)
    AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B').save_pretrained(MERGED)
    print('PeFT merge complete ->', MERGED)

# ---------- persist to Drive (best-effort; ~1.2 GB) ----------
try:
    dst = f'{CKPT}/qwen3-0.6b-masld-merged'
    if not os.path.exists(dst):
        shutil.copytree(MERGED, dst)
    print('persisted to Drive:', dst)
except Exception as e:
    print('Drive copy skipped (model is safe at', MERGED, '):', e)

NOTE: root adapter missing => training did not finish.
      Merging the LATEST checkpoint. Check cell 8 output to see why it stopped;
      re-run cell 8 for the final adapter if it crashed.
merging adapter: /content/drive/Othercomputers/My Mac/masld-llm/checkpoints/qwen3-0.6b-masld-sft/checkpoint-225
--- llamafactory export FAILED, stderr (tail) ---
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/_ops.py", line 1503, in load_library
    ctypes.CDLL(path)
    ~~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/ctypes/__init__.py", line 361, in __init__
    self._handle = self._load_library(name, mode, handle, winmode)
                   ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/ctypes/__init__.py", line 403, in _load_library
    return _dlopen(name, mode)
OSError: libcudart.so.13: cannot open shared object file: No such file or directory

The above exception was the direct cause of the following exception:

Traceback

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

... (4 outputs left out)

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

PeFT merge complete -> /content/qwen3-0.6b-masld-merged
persisted to Drive: /content/drive/Othercomputers/My Mac/masld-llm/checkpoints/qwen3-0.6b-masld-merged


### vLLM install + evaluation


In [22]:
# NOTE: vLLM 0.30 on Colab may fail with 'libcudart.so.13 not found' - the base
# image ships CUDA-12 RAPIDS libs while vllm links CUDA 13. This cell tries the
# vLLM fast path (with a preload fix) and falls back to transformers batch
# generation, which needs no extra CUDA libs and is fast enough for a 0.6B model.
import gzip, json, random, re, os, glob

model_path = ('/content/qwen3-0.6b-masld-merged'
              if os.path.exists('/content/qwen3-0.6b-masld-merged')
              else f'{CKPT}/qwen3-0.6b-masld-merged')
assert os.path.exists(f'{model_path}/config.json'), f'merged model missing at {model_path} - re-run cell 10'
print('evaluating model:', model_path)

def parse_answer(text, task):
    t = (text or '').strip().lower()
    if task.endswith(('001', '201')):
        if t.startswith('yes') or 'answer: yes' in t or 'answer is yes' in t: return 'yes'
        if t.startswith('no') or 'answer: no' in t or 'answer is no' in t: return 'no'
        return None
    if task.endswith(('003', '203')):
        if re.search(r'\banswer(?: is|:) a\b', t) or t.startswith('a'): return 'A'
        if re.search(r'\banswer(?: is|:) b\b', t) or t.startswith('b'): return 'B'
        return None
    m = re.search(r'\bf([0-4])\b', t)
    if m: return f'F{m.group(1)}'
    m = re.search(r'([0-9]+(?:\.[0-9]+)?)', t)
    if m:
        try: return float(m.group(1))
        except ValueError: return None
    return None

rng = random.Random(20260929)
rows = []
with gzip.open(f'{PROMPTS}/MB-20xx_liver_test.jsonl.gz', 'rt') as f:
    f.readline()
    for line in f:
        rows.append(json.loads(line))
by_task = {}
for r in rows:
    by_task.setdefault(r['task'], []).append(r)
sample = []
for task, items in sorted(by_task.items()):
    rng.shuffle(items)
    sample.extend(items[:300])
print('eval prompts:', len(sample))

raws = None

# ---------- path 1: vLLM with libcudart preload fix ----------
try:
    try:
        from vllm import LLM, SamplingParams
    except ImportError as e:
        if 'libcudart' not in str(e):
            raise
        import ctypes, subprocess
        cands = (glob.glob('/usr/local/lib/python3.*/dist-packages/nvidia/*/lib/libcudart.so.13')
                 + glob.glob('/usr/local/cuda*/lib64/libcudart.so.13'))
        if not cands:
            cands = subprocess.run(
                ['find', '/usr/local/lib/python3.13/dist-packages', '-name', 'libcudart.so.13'],
                capture_output=True, text=True).stdout.split()
        if not cands:
            raise RuntimeError('libcudart.so.13 not found anywhere')
        ctypes.CDLL(cands[0], mode=ctypes.RTLD_GLOBAL)
        from vllm import LLM, SamplingParams
    llm = LLM(model=model_path, gpu_memory_utilization=0.9)
    sp = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=200)
    outs = llm.generate([r['prompt'] for r in sample], sp)
    raws = [o.outputs[0].text for o in outs]
    print('vLLM path used')
except Exception as e:
    print('vLLM path failed:', str(e)[:200])

# ---------- path 2: transformers batched greedy generation ----------
if raws is None:
    print('=== using transformers fallback ===')
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_path, padding_side='left')
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16, device_map='cuda')
    model.eval()
    raws = []
    BATCH = 8
    i = 0
    while i < len(sample):
        batch = sample[i:i+BATCH]
        try:
            enc = tok([r['prompt'] for r in batch], return_tensors='pt',
                      padding=True, truncation=True, max_length=4096).to('cuda')
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=200, do_sample=False,
                                     pad_token_id=tok.pad_token_id)
            for j in range(len(batch)):
                raws.append(tok.decode(gen[j][enc['input_ids'].shape[1]:],
                                       skip_special_tokens=True))
            i += len(batch)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if BATCH == 1:
                raise
            BATCH = max(1, BATCH // 2)
            print(f'OOM -> batch reduced to {BATCH}, restarting this chunk')
        if (i // BATCH) % 10 == 0 and i % 80 < BATCH:
            print(f'  {i}/{len(sample)}', flush=True)
    print('transformers path complete')

# ---------- archive (identical format to frontier runs) ----------
runs_dir = (os.path.join(MASLD_LLM, 'results', 'frontier_runs')
            if os.access(os.path.join(MASLD_LLM, 'results'), os.W_OK)
            else '/content/drive/MyDrive/masld-llm-outputs/frontier_runs')
os.makedirs(runs_dir, exist_ok=True)
run_path = runs_dir + '/qwen3-0.6b-masld__MB-20xx.jsonl.gz'
with gzip.open(run_path, 'wt') as f:
    for r, raw in zip(sample, raws):
        f.write(json.dumps(dict(task=r['task'], sample=r['sample'], answer=r['answer'],
                                raw=raw, error=None, model_version='qwen3-0.6b-masld-sft',
                                parsed=parse_answer(raw, r['task']))) + chr(10))
print('archived ->', run_path)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

ImportError: libcudart.so.13: cannot open shared object file: No such file or directory

## 4 · Ladder continuation

- **1.7B**: same config, `model_name_or_path: Qwen/Qwen3-1.7B`.

- **9B two-stage QLoRA**: A100 40GB (Pro+). Stage A = KG/drug text, 1 epoch, lr 2e-5;

  stage B = this config from the stage-A adapter, lr 1e-4, `quantization_bit: 4`.

- After each run: sync home, then locally `python3 scripts/score_frontier.py`.
